# Face Tracker — Google Colab
**Before running:** Runtime → Change runtime type → T4 GPU → Save

Run each cell in order.

In [ ]:
# Cell 1: Check GPU
import subprocess
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode == 0:
    print('GPU ready:', r.stdout.strip())
else:
    print('NO GPU detected → Runtime > Change runtime type > T4 GPU')
    raise SystemExit('Need GPU')

GPU ready: Tesla T4, 15360 MiB


In [ ]:
# Cell 2: Install dependencies
!pip install ultralytics insightface onnxruntime-gpu deep_sort_realtime \
             flask opencv-python-headless Pillow scipy -q
print('Dependencies installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 14.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 74.0 MB/s eta 0:00:00
Dependencies installed.


In [ ]:
# Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [ ]:
# Cell 4: Set project path (edit to match your Drive folder name)
import os
PROJECT = '/content/drive/MyDrive/6th sem /Company hackathons/face_tracker_final'
os.chdir(PROJECT)
print('Working directory:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

Working directory: /content/drive/MyDrive/6th sem /Company hackathons/face_tracker_final
Files: ['README.md', '__pycache__', 'app.py', 'commands.txt', 'config', 'face_tracker.db', 'inspect_db.py', 'logs', 'main.py', 'requirements.txt', 'src', 'verify.py', 'video_sample1.mp4', 'yolov8n-face.pt', 'yolov8n.pt']


In [ ]:
# Cell 5: Download YOLO model (once)
import os, shutil
from ultralytics import YOLO
if not os.path.exists('yolov8n.pt'):
    YOLO('yolov8n.pt')
if not os.path.exists('yolov8n-face.pt'):
    shutil.copy('yolov8n.pt', 'yolov8n-face.pt')
print('Model ready:', round(os.path.getsize('yolov8n.pt')/1024/1024,1), 'MB')

Model ready: 6.2 MB


In [ ]:
# Cell 6: Reset for fresh run (set RESET=False to keep previous data)
import os, shutil
RESET = True
if RESET:
    for p in ['face_tracker.db']:
        if os.path.exists(p): os.remove(p)
    for d in ['logs/entries','logs/exits','logs/registered']:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)
    open('logs/events.log','w').close()
    print('Fresh start: DB and logs cleared.')
else:
    print('Keeping existing data.')

Fresh start: DB and logs cleared.


In [ ]:
# cell 7 - QUICK DIAGNOSTIC — run this in a new cell
import subprocess, time, requests

# Check if Flask is running on port 5000
try:
    r = requests.get('http://localhost:5000', timeout=3)
    print(f"Flask IS running: status {r.status_code}")
except Exception as e:
    print(f"Flask NOT running: {e}")

# Check if cloudflared is running
result = subprocess.run(['pgrep', '-a', 'cloudflared'],
                       capture_output=True, text=True)
print(f"cloudflared process: {result.stdout.strip() or 'NOT RUNNING'}")

# Check what's on port 5000
result = subprocess.run(['ss', '-tlnp'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '5000' in line:
        print(f"Port 5000: {line}")

INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:27:37] "GET / HTTP/1.1" 200 -


Flask IS running: status 200
cloudflared process: 4000 cloudflared
Port 5000: LISTEN 0      128          0.0.0.0:5000       0.0.0.0:*    users:(("python3",pid=3530,fd=49))     


In [ ]:
# cell 8 KILL CELL — run this BEFORE Cell 7 every time
import subprocess, time

# Kill everything on port 5000
subprocess.run('fuser -k 5000/tcp 2>/dev/null || true', shell=True)
subprocess.run('pkill -f "main.py" 2>/dev/null || true', shell=True)
subprocess.run('pkill -f cloudflared 2>/dev/null || true', shell=True)
time.sleep(3)

# Verify port is free
result = subprocess.run('ss -tlnp | grep 5000', shell=True, capture_output=True, text=True)
if result.stdout.strip():
    print("WARNING: port 5000 still in use:", result.stdout.strip())
else:
    print("Port 5000 is free. Now run Cell 9.")

Port 5000 is free. Now run Cell 9.


In [ ]:
# ============================================================
# COLAB CELL 9 — Copy this entire cell into your notebook
# ============================================================
# This replaces the previous Cell 7.
# It starts Flask + tunnel in the correct order.
# ============================================================

import subprocess, threading, time, re, sys, os, json, socket

PORT = 5000
PROJECT = '/content/drive/MyDrive/6th sem /Company hackathons/face_tracker_final'  # edit if needed

# Step 0: Go to project dir
os.chdir(PROJECT)
sys.path.insert(0, os.path.join(PROJECT, 'src'))

# Step 1: Patch config (no display window in Colab)
with open('config/config.json') as f:
    cfg = json.load(f)
cfg['display']['show_window'] = False
with open('config/config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

# Step 2: Kill anything using port 5000
subprocess.run(f'fuser -k {PORT}/tcp 2>/dev/null || true', shell=True)
subprocess.run('pkill -f cloudflared 2>/dev/null || true', shell=True)
time.sleep(2)

# Step 3: Import main module and initialise globals
# We import the module directly so _CFG gets set in the same process
import importlib.util
spec = importlib.util.spec_from_file_location("face_tracker_main", "main.py")
M = importlib.util.module_from_spec(spec)
spec.loader.exec_module(M)

# Initialise config and logging (sets M._CFG)
M._init_globals(config_path='config/config.json', port=PORT)
print("Config loaded:", M._CFG is not None)

# Step 4: Start Flask in background thread
flask_thread = threading.Thread(
    target=lambda: M.flask_app.run(
        host='0.0.0.0', port=PORT,
        debug=False, use_reloader=False
    ),
    daemon=True
)
flask_thread.start()

# Step 5: Wait until Flask is actually accepting connections
print("Waiting for Flask...", end='', flush=True)
deadline = time.time() + 25
flask_ok = False
while time.time() < deadline:
    try:
        s = socket.create_connection(('localhost', PORT), timeout=1)
        s.close()
        flask_ok = True
        print(" Ready!")
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(0.5)

if not flask_ok:
    print("\nFlask failed to start!")
    raise RuntimeError("Flask did not bind to port 5000")

# Step 6: Install + start Cloudflare tunnel
if subprocess.run(['which','cloudflared'], capture_output=True).returncode != 0:
    print("Installing cloudflared...")
    subprocess.run(
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
        'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && '
        'chmod +x /usr/local/bin/cloudflared',
        shell=True, check=True
    )
    print("cloudflared installed.")

tunnel_proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

# Step 7: Read tunnel output until URL appears
url = ''
pat = re.compile(r'https://[a-z0-9\-]+\.trycloudflare\.com')
deadline = time.time() + 45
print("Getting public URL", end='', flush=True)
while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    print('.', end='', flush=True)
    m = pat.search(line)
    if m:
        url = m.group(0)
        break

print()
if url:
    print(f"\n{'='*62}")
    print("  Face Tracker Dashboard — OPEN IN YOUR BROWSER:")
    print(f"  {url}")
    print(f"\n  Entries:    {url}/?tab=entries")
    print(f"  Registered: {url}/?tab=registered")
    print(f"  All Events: {url}/?tab=all")
    print(f"\n  Use the control panel on the page to:")
    print("  - Select 'Video File' → type filename → Start")
    print("  - Select 'RTSP Camera' → type rtsp://... → Start")
    print("  - Pause / Resume / Stop / Clear Data")
    print(f"{'='*62}")
    print("\n  Keep this cell running. Interrupt it to shut down.")
else:
    print("Tunnel URL not captured. Raw output:")
    for _ in range(20):
        l = tunnel_proc.stdout.readline()
        if l: print(" ", l.strip())

# Step 8: Keep alive (tunnel dies if cell stops)
try:
    while True:
        time.sleep(10)
        # Check Flask is still alive
        try:
            s = socket.create_connection(('localhost', PORT), timeout=1)
            s.close()
        except Exception:
            print("Flask died — restarting...")
            flask_thread = threading.Thread(
                target=lambda: M.flask_app.run(
                    host='0.0.0.0', port=PORT,
                    debug=False, use_reloader=False
                ),
                daemon=True
            )
            flask_thread.start()
            time.sleep(3)
except KeyboardInterrupt:
    tunnel_proc.terminate()
    print("\nDashboard stopped.")

Config loaded: True
Waiting for Flask.... * Serving Flask app 'face_tracker_main'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


 Ready!
Getting public URL.....

  Face Tracker Dashboard — OPEN IN YOUR BROWSER:
  https://assists-testimonials-vat-extended.trycloudflare.com

  Entries:    https://assists-testimonials-vat-extended.trycloudflare.com/?tab=entries
  Registered: https://assists-testimonials-vat-extended.trycloudflare.com/?tab=registered
  All Events: https://assists-testimonials-vat-extended.trycloudflare.com/?tab=all

  Use the control panel on the page to:
  - Select 'Video File' → type filename → Start
  - Select 'RTSP Camera' → type rtsp://... → Start
  - Pause / Resume / Stop / Clear Data

  Keep this cell running. Interrupt it to shut down.


INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:51:10] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:51:11] "GET /img?p=logs/entries/2026-03-23/face_20260323_034312_1839fa_034401_004.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:51:11] "GET /img?p=logs/entries/2026-03-23/face_20260323_034310_5a5e80_034347_733.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:51:11] "GET /img?p=logs/entries/2026-03-23/face_20260323_034304_a837b8_034341_059.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:51:11] "GET /img?p=logs/entries/2026-03-23/face_20260323_034312_1839fa_034340_129.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:51:11] "GET /img?p=logs/entries/2026-03-23/face_20260323_034329_086706_034400_056.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:51:11] "GET /img?p=logs/entries/2026-03-23/face_20260323_034304_281858_034348_903.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:51:11]

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:52:05] "GET /?tab=entries HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:52:09] "GET /?tab=entries HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:52:10] "GET /img?p=logs/entries/2026-03-23/face_20260323_035206_aba66d_035207_430.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:52:10] "GET /img?p=logs/entries/2026-03-23/face_20260323_035206_97e186_035207_416.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:52:10] "GET /img?p=logs/entries/2026-03-23/face_20260323_035206_71cad9_035207_490.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:52:10] "GET /img?p=logs/entries/2026-03-23/face_20260323_035207_d80d41_035207_570.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:52:10] "GET /img?p=logs/entries/2026-03-23/face_20260323_035206_429f96_035207_473.jpg HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Mar/2026 03:52:10] "GET /img?p=logs/entries/2026-03-23/face_202603


Dashboard stopped.


In [ ]:
# Cell 10: Verify results
!python verify.py


FACE TRACKER — POST-RUN VERIFICATION

✗ Database not found: face_tracker.db
  Run main.py first to process a video.


In [ ]:
# Cell 11: View face crop images inline
import os
from IPython.display import display, Image as IPImage
for label, folder in [('REGISTERED','logs/registered'),('ENTRIES','logs/entries')]:
    imgs = []
    for root,_,files in os.walk(folder):
        imgs += [os.path.join(root,f) for f in sorted(files) if f.endswith('.jpg')]
    print(f'--- {label}: {len(imgs)} total, showing first 12 ---')
    for p in imgs[:12]: display(IPImage(filename=p, width=112))
    print()

## Alternative: Colab built-in port forwarding
If the Cloudflare tunnel doesn't work:
1. Look at the bottom of the Colab screen for a **Ports** tab
2. Click it, then click **Add port**, type `5000`, press Enter
3. Click the **link icon** that appears — this opens the dashboard in your browser

This is Google's own port forwarding and always works in Colab Pro/Pro+.